# Caso 3: AVANZADO - Calidad de Datos RENAMU 2022 (Registro Nacional de Municipalidades)

---

## Contexto del Proyecto

### Descripción del Problema
El **Instituto Nacional de Estadística e Informática (INEI)**, a través del
**RENAMU** (Registro Nacional de Municipalidades), recopila información
administrativa, de recursos humanos, servicios y gestión de las municipalidades
peruanas. Al consolidar los datos del año 2022, se detectan valores faltantes,
inconsistencias en códigos geográficos y registros duplicados que impiden
generar análisis confiables por municipalidad.

### Objetivo Analítico
Implementar un pipeline robusto de calidad sobre los datos del RENAMU 2022 para:
- Detectar y corregir automáticamente Ubigeos inválidos o inconsistentes
- Registrar cada problema en un log de auditoría documentado
- Generar métricas de calidad cuantificables antes y después de la limpieza
- Producir el archivo Silver reproducible para el JOIN con SIAF y SISMEPRE

### Impacto de la Mala Calidad de Datos
- **Financiero**: Un Ubigeo incorrecto rompe el JOIN con SIAF y SISMEPRE
- **Operativo**: Duplicados por municipalidad generan conteos incorrectos
- **Estratégico**: Decisiones de política pública basadas en datos
  incompletos afectan la asignación de recursos a municipios

---

## Dimensiones de Calidad a Corregir

1. **Completitud**: Imputar nulos en Departamento, Provincia y Distrito con logging
2. **Exactitud**: Reconstruir Ubigeo desde ccdd+ccpp+ccdi con registro en auditoría
3. **Consistencia**: Validación automática entre Ubigeo y códigos geográficos
4. **Integridad**: Validación de ccdd en rango 01-25
5. **Razonabilidad**: Validación de Tipomuni en rango 1-5
6. **Oportunidad**: Validación automática del año 2022
7. **Unicidad**: Deduplicación por Ubigeo con logging
8. **Validez**: Pipeline de validación automática del formato Ubigeo

---

In [1]:
# Instalación de librerías necesarias
# !pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline


In [3]:
# Cargar datos RENAMU 2022
df_avanzado = pd.read_csv('Base_RENAMU_2022_f.csv', encoding='latin-1', sep=';')

In [4]:
# Convertir Ubigeo a string desde el inicio
df_avanzado['Ubigeo'] = df_avanzado['Ubigeo'].astype(str).str.zfill(6)

print(f"✅ Dataset RENAMU 2022 cargado correctamente")


✅ Dataset RENAMU 2022 cargado correctamente


---

# SOLUCIÓN NIVEL AVANZADO - RENAMU 2022

## Objetivo
Implementar un pipeline robusto y automatizado de calidad de datos:
- Validaciones basadas en reglas del catálogo geográfico del INEI
- Reconstrucción automática del Ubigeo desde ccdd+ccpp+ccdi
- Logging de problemas para auditoría
- Métricas de calidad cuantificables
- Proceso reproducible y escalable

In [5]:
quality_log = []
print(f"\nRegistros iniciales: {len(df_avanzado):,}")


Registros iniciales: 1,874


In [6]:
class DataQualityValidator:

    def __init__(self, df):
        self.df = df.copy()
        # Convertir Ubigeo a string desde el inicio
        self.df['Ubigeo'] = self.df['Ubigeo'].astype(str).str.zfill(6)
        self.initial_count = len(df)
        self.now = datetime.now()
        self.quality_log = []

    # LOGGING PROFESIONAL
    def log_issue(self, dimension, rule, affected_rows, action):
        self.quality_log.append({
            "timestamp":     self.now,
            "dimension":     dimension,
            "rule":          rule,
            "affected_rows": int(affected_rows),
            "action":        action
        })

    # 1. COMPLETITUD
    def validate_completeness(self):
        # Columnas críticas: eliminar si son nulas
        criticas = ['Ubigeo', 'Departamento', 'Provincia', 'Distrito']
        for col in criticas:
            mask  = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                self.df = self.df.loc[~mask]
                self.log_issue("completitud", f"{col}_null", count, "drop")

        # Imputar con moda
        for col in ['Departamento', 'Provincia', 'Distrito']:
            mask  = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                fill = self.df[col].mode().iloc[0]
                self.df[col] = self.df[col].fillna(fill)
                self.log_issue("completitud", f"{col}_null", count, f"fill_mode:{fill}")

    # 2. EXACTITUD
    def validate_accuracy(self):
        # Reconstruir Ubigeo inválido desde ccdd+ccpp+ccdi
        self.df['Ubigeo_calculado'] = (
            self.df['ccdd'].astype(str).str.zfill(2) +
            self.df['ccpp'].astype(str).str.zfill(2) +
            self.df['ccdi'].astype(str).str.zfill(2)
        )
        mask  = ~self.df['Ubigeo'].str.match(r'^\d{6}$')
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'Ubigeo'] = self.df.loc[mask, 'Ubigeo_calculado']
            self.log_issue("exactitud", "Ubigeo_invalido", count,
                           "reconstruido_desde_ccdd_ccpp_ccdi")

        # Tipomuni fuera de rango
        self.df['Tipomuni'] = pd.to_numeric(self.df['Tipomuni'], errors='coerce')
        mask  = ~self.df['Tipomuni'].isin([1, 2, 3, 4, 5])
        count = mask.sum()
        if count > 0:
            self.df['Tipomuni'] = self.df['Tipomuni'].astype(float)
            moda_tipo = self.df[~mask]['Tipomuni'].mode().iloc[0]
            self.df.loc[mask, 'Tipomuni'] = float(moda_tipo)
            self.log_issue("exactitud", "Tipomuni_invalido", count,
                           f"fill_moda:{moda_tipo}")

    # 3. CONSISTENCIA
    def validate_consistency(self):
        # Corregir Ubigeo inconsistente con ccdd+ccpp+ccdi
        if 'Ubigeo_calculado' not in self.df.columns:
            self.df['Ubigeo_calculado'] = (
                self.df['ccdd'].astype(str).str.zfill(2) +
                self.df['ccpp'].astype(str).str.zfill(2) +
                self.df['ccdi'].astype(str).str.zfill(2)
            )
        mask  = self.df['Ubigeo'] != self.df['Ubigeo_calculado']
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'Ubigeo'] = self.df.loc[mask, 'Ubigeo_calculado']
            self.log_issue("consistencia", "Ubigeo_vs_codigos", count, "corrected")

        self.df.drop(columns=['Ubigeo_calculado'], inplace=True, errors='ignore')

        # Estandarizar nombres geográficos
        for col in ['Departamento', 'Provincia', 'Distrito']:
            self.df[col] = self.df[col].str.upper().str.strip()

    # 4. INTEGRIDAD
    def validate_integrity(self):
        # ccdd fuera de rango 01-25
        mask  = (self.df['ccdd'] < 1) | (self.df['ccdd'] > 25)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("integridad", "ccdd_fuera_rango", count, "drop")

        # idmunici nulo
        mask  = self.df['idmunici'].isnull()
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("integridad", "idmunici_nulo", count, "drop")

    # 5. RAZONABILIDAD
    def validate_reasonableness(self):
        # Tipomuni debe estar entre 1 y 5
        mask  = ~self.df['Tipomuni'].isin([1, 2, 3, 4, 5])
        count = mask.sum()
        if count > 0:
            self.log_issue("razonabilidad", "Tipomuni_fuera_rango",
                           count, "logged_only")

    # 6. OPORTUNIDAD
    def validate_timeliness(self):
        col_anio = [c for c in self.df.columns
                    if 'o' in c.lower() or 'a' in c.lower()[:2]][0]
        mask  = self.df[col_anio] != 2022
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("oportunidad", "anio_diferente_2022", count, "drop")

    # 7. UNICIDAD
    def validate_uniqueness(self):
        # Duplicados exactos
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        count  = before - len(self.df)
        if count > 0:
            self.log_issue("unicidad", "duplicados_exactos", count, "drop")

        # Duplicados por Ubigeo
        before = len(self.df)
        self.df = self.df.drop_duplicates(subset=['Ubigeo'], keep='first')
        count  = before - len(self.df)
        if count > 0:
            self.log_issue("unicidad", "duplicados_ubigeo", count, "drop")

    # 8. VALIDEZ
    def validate_validity(self):
        # Ubigeo debe tener exactamente 6 dígitos numéricos
        mask  = ~self.df['Ubigeo'].str.match(r'^\d{6}$')
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("validez", "Ubigeo_formato_invalido", count, "drop")

    # PIPELINE
    def run(self):
        print("\n🔄 Ejecutando pipeline de calidad RENAMU 2022...")
        self.validate_completeness()
        print("   ✅ Completitud")
        self.validate_accuracy()
        print("   ✅ Exactitud")
        self.validate_consistency()
        print("   ✅ Consistencia")
        self.validate_integrity()
        print("   ✅ Integridad")
        self.validate_reasonableness()
        print("   ✅ Razonabilidad")
        self.validate_timeliness()
        print("   ✅ Oportunidad")
        self.validate_uniqueness()
        print("   ✅ Unicidad")
        self.validate_validity()
        print("   ✅ Validez")
        print("\n✅ Pipeline completado")
        return self.df

    # MÉTRICAS
    def report(self):
        final_count = len(self.df)
        perdida     = self.initial_count - final_count
        pct_perdida = (perdida / self.initial_count * 100) if self.initial_count > 0 else 0

        print("\n" + "=" * 60)
        print("   REPORTE FINAL - PIPELINE AVANZADO RENAMU 2022")
        print("=" * 60)
        print(f"\n📊 Registros iniciales : {self.initial_count:,}")
        print(f"📊 Registros finales   : {final_count:,}")
        print(f"📊 Registros perdidos  : {perdida:,} ({pct_perdida:.2f}%)")

        log_df = pd.DataFrame(self.quality_log)
        if len(log_df) > 0:
            print("\n📋 Resumen por dimensión:")
            print("-" * 60)
            print(log_df.groupby("dimension")["affected_rows"].sum().to_string())
            print("-" * 60)
            print(f"\n📋 Log completo de auditoría:")
            print(log_df[['dimension', 'rule',
                           'affected_rows', 'action']].to_string(index=False))
        else:
            print("\n✅ No se registraron problemas en el log")

        return log_df

In [7]:
# Ejecutar pipeline avanzado de calidad RENAMU 2022
validator = DataQualityValidator(df_avanzado)
df_clean  = validator.run()
log_df    = validator.report()


🔄 Ejecutando pipeline de calidad RENAMU 2022...
   ✅ Completitud
   ✅ Exactitud
   ✅ Consistencia
   ✅ Integridad
   ✅ Razonabilidad
   ✅ Oportunidad
   ✅ Unicidad
   ✅ Validez

✅ Pipeline completado

   REPORTE FINAL - PIPELINE AVANZADO RENAMU 2022

📊 Registros iniciales : 1,874
📊 Registros finales   : 1,874
📊 Registros perdidos  : 0 (0.00%)

✅ No se registraron problemas en el log


In [8]:
# Generar reporte final
log_df

""


In [9]:
# Convertir columnas object a string para compatibilidad con Parquet
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].astype(str)

df_clean.to_parquet("renamu_silver_avanzado.parquet", index=False)
print(f"✅ Archivo guardado: renamu_silver_avanzado.parquet")
print(f"   Registros: {len(df_clean):,}")
print(f"   Columnas : {len(df_clean.columns)}")

✅ Archivo guardado: renamu_silver_avanzado.parquet
   Registros: 1,874
   Columnas : 1368


### Conclusiones de la Solución Avanzada - RENAMU 2022

**Ventajas:**
- Pipeline automatizado y reproducible aplicable a futuros años del RENAMU
- Logging completo para auditoría: cada problema queda registrado con
  dimensión, regla, cantidad de registros afectados y acción tomada
- Trata las 8 dimensiones de calidad sobre datos reales del RENAMU
- Reconstruye Ubigeos inválidos en lugar de eliminarlos directamente
- Trazabilidad completa de cada decisión de limpieza

**Características clave:**
- Modular: cada dimensión es un método independiente que puede
  modificarse sin afectar el resto del pipeline
- Métricas cuantificables: el reporte final muestra pérdida exacta
  de registros por dimensión
- Reglas de negocio explícitas del catálogo INEI (Tipomuni 1-5,
  ccdd 01-25, Ubigeo 6 dígitos, año 2022)
- Escalable: la clase DataQualityValidator puede instanciarse
  con el RENAMU de cualquier otro año sin cambiar la estructura